# 02 - Ingeniería de características y datasets de modelado por horizonte

Este notebook transforma la serie horaria procesada de O3 en los conjuntos de datos que se utilizarán durante la fase de modelado de acuerdo con la siguiente lógica metodológica:

$$
X_t \rightarrow y_{t+h}
$$

Donde:

- `origin_timestamp = t` es el instante desde el que se formula la predicción
- `target_timestamp = t + h` es el instante que se desea predecir;
- `y` es la concentración de O3 observada en el instante objetivo;
- `h` es el horizonte de predicción, en horas.

Un punto fundamental es evitar que se produzcan fugas temporales, por lo que las variables basadas en O3 solo pueden utilizar valores observados en `t` o antes. Las variables de calendario se calcularán sobre el instante objetivo, porque la hora, el día de la semana y el mes futuros son información conocida con anterioridad.

## Objetivos del notebook

Este notebook tiene los siguientes objetivos:

1. Cargar el conjunto de datos maestro generado durante la preparación de la serie horaria.
2. Establecer la partición cronológica del proyecto: entrenamiento, calibración, validación y prueba.
3. Construir un conjunto de variables predictoras derivado de la EDA.
4. Generar datasets de modelado para los horizontes de 1, 4, 12 y 24 horas.
5. Guardar los datasets y las tablas de control necesarias para los futuros notebooks de entrenamiento.

In [1]:

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

In [ ]:

def find_project_root(start: Path | None = None) -> Path:
    """Busca la raíz del proyecto partiendo del directorio actual.

    Esta función evita depender de rutas absolutas locales. Así, el notebook puede
    ejecutarse aunque el repositorio esté guardado en otra carpeta o se abra desde
    un directorio diferente dentro del proyecto.
    """
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
    )

PROJECT_ROOT = find_project_root()

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "o3_hourly.parquet"
MODELING_DATA_DIR = PROJECT_ROOT / "data" / "modeling"
REPORTS_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

MODELING_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Dataset procesado: {PROCESSED_DATA_PATH}")
print(f"Directorio de modelado: {MODELING_DATA_DIR}")
print(f"Directorio de tablas: {REPORTS_TABLES_DIR}")

Raíz del proyecto: C:\trabajo_github
Dataset procesado: C:\trabajo_github\data\processed\o3_hourly.parquet
Directorio de modelado: C:\trabajo_github\data\modeling
Directorio de tablas: C:\trabajo_github\reports\tables


In [3]:
# Configuración metodológica del proyecto.

TARGET_COLUMN = "o3"

# Horizontes utilizados en la comparación inicial de modelos.
# El modelo seleccionado se extenderá posteriormente al rango completo 1-24 horas.
HORIZONS = [1, 4, 12, 24]

# Partición cronológica tal y como se establece en la memoria.
# La asignación de cada fila se realizará a partir del instante objetivo,
# no del origen de predicción.
SPLIT_DATE_RANGES = {
    "train": (pd.Timestamp("2020-01-01 00:00:00"), pd.Timestamp("2022-12-31 23:00:00")),
    "calibration": (pd.Timestamp("2023-01-01 00:00:00"), pd.Timestamp("2023-12-31 23:00:00")),
    "validation": (pd.Timestamp("2024-01-01 00:00:00"), pd.Timestamp("2024-12-31 23:00:00")),
    "test": (pd.Timestamp("2025-01-01 00:00:00"), pd.Timestamp("2025-12-31 23:00:00")),
}

# Conjunto de variables.
# Se decide en este momento para que, posteriormente todos los modelos del TFG trabajen sobre una base común,
# reduciendo posibles redundancias y facilitando la interpretación posterior.
ORIGIN_LAG_HOURS = [1, 2, 3, 6, 12]
TARGET_LAG_HOURS = [48]  # El retardo de 24 h se guarda como o3_prev_day_same_hour.
ROLLING_WINDOWS = [6, 24]

# Variables concretas que se calcularán para cada ventana.
ROLLING_STATS = ["mean", "std"]
INCLUDE_ROLLING_MAX_24H = True
INCLUDE_MISSING_RATIO_24H = True

print("Horizontes:", HORIZONS)
print("Valor en el origen: o3_at_origin")
print("Retardos respecto al origen:", ORIGIN_LAG_HOURS)
print("Retardos diarios adicionales respecto al objetivo:", TARGET_LAG_HOURS)
print("Ventanas móviles:", ROLLING_WINDOWS)

Horizontes: [1, 4, 12, 24]
Valor en el origen: o3_at_origin
Retardos respecto al origen: [1, 2, 3, 6, 12]
Retardos diarios adicionales respecto al objetivo: [48]
Ventanas móviles: [6, 24]


## 1. Carga y validación básica del conjunto procesado

El notebook parte del archivo `data/processed/o3_hourly.parquet`, generado en la fase de preparación de la serie horaria.

Se comprueba que el conjunto contenga, como mínimo:

- `timestamp`: la marca temporal horaria
- `o3`: la concentración horaria de ozono troposférico

In [4]:

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        "No se ha encontrado data/processed/o3_hourly.parquet. "
        "Debe ejecutarse con anterioridad el notebook de preparación de la serie horaria."
    )

hourly_df = pd.read_parquet(PROCESSED_DATA_PATH).copy()

required_columns = {"timestamp", TARGET_COLUMN}
missing_columns = required_columns - set(hourly_df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas obligatorias en el conjunto procesado: {missing_columns}")

hourly_df["timestamp"] = pd.to_datetime(hourly_df["timestamp"])
hourly_df = hourly_df.sort_values("timestamp").reset_index(drop=True)

print(f"Filas: {hourly_df.shape[0]:,}")
print(f"Columnas: {hourly_df.shape[1]:,}")
print(f"Inicio: {hourly_df['timestamp'].min()}")
print(f"Fin: {hourly_df['timestamp'].max()}")
print(f"Valores ausentes de {TARGET_COLUMN}: {hourly_df[TARGET_COLUMN].isna().sum():,}")

display(hourly_df.head())
display(hourly_df.tail())

Filas: 52,608
Columnas: 16
Inicio: 2020-01-01 00:00:00
Fin: 2025-12-31 23:00:00
Valores ausentes de o3: 1,666


,timestamp,o3,station_code,station_name,pollutant_code,pollutant,units,station_type,urban_area,ine_code,municipality,county_code,county_name,altitude,latitude,longitude
0,2020-01-01 00:00:00,22.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538
1,2020-01-01 01:00:00,19.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538
2,2020-01-01 02:00:00,19.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538
3,2020-01-01 03:00:00,6.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538
4,2020-01-01 04:00:00,8.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538


,timestamp,o3,station_code,station_name,pollutant_code,pollutant,units,station_type,urban_area,ine_code,municipality,county_code,county_name,altitude,latitude,longitude
52603,2025-12-31 19:00:00,25.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538
52604,2025-12-31 20:00:00,32.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538
52605,2025-12-31 21:00:00,36.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538
52606,2025-12-31 22:00:00,39.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538
52607,2025-12-31 23:00:00,42.0,8019043,Barcelona (Eixample),14,O3,µg/m3,traffic,urban,8019,Barcelona,13,Barcelonès,26,41.385315,2.1538


In [5]:

# Comprobación de la regularidad horaria.
# La serie debería contener un registro por cada hora del periodo considerado.

timestamp_diff = hourly_df["timestamp"].diff().dropna()
unexpected_diffs = timestamp_diff[timestamp_diff != pd.Timedelta(hours=1)]

if not unexpected_diffs.empty:
    raise ValueError(
        "La serie no parece tener frecuencia horaria regular. "
    )

duplicated_timestamps = hourly_df["timestamp"].duplicated().sum()
if duplicated_timestamps > 0:
    raise ValueError(f"Existen {duplicated_timestamps} timestamps duplicados.")

print("La serie presenta frecuencia horaria regular y no contiene timestamps duplicados.")

La serie presenta frecuencia horaria regular y no contiene timestamps duplicados.


## 2. Funciones auxiliares

Las siguientes funciones se utilizan para:

- Crear codificaciones cíclicas seno-coseno
- Asignar cada fila a un bloque cronológico
- Comprobar que no se esté utilizando información futura
- Documentar filas y valores ausentes por partición.

In [ ]:

def add_cyclical_encoding(df: pd.DataFrame, column: str, period: int, prefix: str) -> pd.DataFrame:
    """Añade la codificación seno-coseno para una variable cíclica.

    Ejemplos:
    - hora del día: periodo 24
    - día de la semana: periodo 7
    - mes del año: periodo 12.

    Esta codificación evita discontinuidades artificiales entre categorías contiguas,
    como las 23:00 y las 00:00 o diciembre y enero.
    """
    angle = 2 * np.pi * df[column] / period
    df[f"{prefix}_sin"] = np.sin(angle)
    df[f"{prefix}_cos"] = np.cos(angle)
    return df

def assign_chronological_split(target_timestamp: pd.Timestamp) -> str | None:
    """Asigna el bloque cronológico a partir del instante objetivo.

    La asignación se realiza con `target_timestamp` porque el objetivo de la fila es
    predecir dicho instante concreto. De esta manera, una predicción cuyo objetivo cae
    en 2023 se reserva para calibración, aunque su origen pueda estar en 2022.
    """
    for split_name, (start, end) in SPLIT_DATE_RANGES.items():
        if start <= target_timestamp <= end:
            return split_name
    return None

def summarize_by_split(df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    """Resume filas, fechas y valores ausentes por partición."""
    rows = []
    for split_name, split_df in df.groupby("split", sort=False):
        rows.append(
            {
                "horizon": horizon,
                "split": split_name,
                "n_rows": len(split_df),
                "origin_min": split_df["origin_timestamp"].min(),
                "origin_max": split_df["origin_timestamp"].max(),
                "target_min": split_df["target_timestamp"].min(),
                "target_max": split_df["target_timestamp"].max(),
                "missing_target": split_df["y"].isna().sum(),
                "missing_baseline_daily": split_df["o3_prev_day_same_hour"].isna().sum(),
            }
        )
    return pd.DataFrame(rows)

def check_no_temporal_leakage(df: pd.DataFrame, horizon: int) -> None:
    """Verifica reglas básicas para evitar la fuga temporal.

    Para cada fila, todas las variables basadas en O3 deben proceder de instantes
    menores o iguales al origen de predicción. Las variables de calendario del objetivo
    sí se considerarán válidas, pues el calendario futuro se conoce de antemano.
    """
    if not (df["target_timestamp"] == df["origin_timestamp"] + pd.to_timedelta(horizon, unit="h")).all():
        raise AssertionError("La relación origin_timestamp + horizon = target_timestamp no se cumple.")

    # La misma hora del día anterior al objetivo debe estar disponible como máximo en el origen.
    prev_day_timestamp = df["target_timestamp"] - pd.Timedelta(hours=24)
    if not (prev_day_timestamp <= df["origin_timestamp"]).all():
        raise AssertionError("La regla de persistencia diaria usa información futura en alguna fila.")

    # Los retardos adicionales respecto al objetivo también deben ser causales.
    for lag in TARGET_LAG_HOURS:
        source_timestamp = df["target_timestamp"] - pd.to_timedelta(lag, unit="h")
        if not (source_timestamp <= df["origin_timestamp"]).all():
            raise AssertionError(f"El retardo objetivo {lag}h usa información futura.")

def remove_exact_duplicate_features(
    df: pd.DataFrame,
    candidate_features: list[str],
) -> tuple[list[str], dict[str, str]]:
    """Elimina duplicados exactos dentro de un horizonte.

    Algunos predictores pueden coincidir exactamente en determinados horizontes.
    Por ejemplo, para h=24, la misma hora del día anterior al objetivo coincide
    con el valor observado en el origen. En esos casos se elimina la segunda columna
    para evitar "redundancia perfecta" en la regresión lineal.
    """
    selected_features: list[str] = []
    dropped_duplicates: dict[str, str] = {}

    for feature in candidate_features:
        duplicate_of = None

        for selected in selected_features:
            if df[feature].equals(df[selected]):
                duplicate_of = selected
                break

        if duplicate_of is None:
            selected_features.append(feature)
        else:
            dropped_duplicates[feature] = duplicate_of

    return selected_features, dropped_duplicates

## 3. Variables calculadas en el origen de predicción

En este apartado y en primer lugar, se construirán variables que dependan exclusivamente de la información observada hasta el origen `t`.

Estas variables se calculan una sola vez sobre la serie horaria:

- Valor observado en el origen de predicción (`o3_at_origin`)
- Retardos recientes respecto al origen (`o3_lag_1h`, `o3_lag_2h`, etc.)
- Medias móviles de 6 y 24 horas
- Desviaciones móviles de 6 y 24 horas
- Máximo móvil de 24 horas;
- Proporción de ausencias en la ventana de 24 horas.

No se elimina ningún valor ausente en esta fase. Cuando se programe el `pipeline` de los modelos posteriores, se decidirá si en ese caso deben imputarse o si, como en este caso, pueden gestionar los `NaN` directamente.

In [7]:
# Se trabaja con una vista de la serie indexada por timestamp.
base_df = hourly_df[["timestamp", TARGET_COLUMN]].copy()
base_df = base_df.sort_values("timestamp").reset_index(drop=True)

# Valor observado en el origen de predicción.
# Esta columna representa y_t cuando el objetivo es y_{t+h}.
# No se denomina lag_0 para evitar confundirla con un valor pasado respecto al origen.
base_df["o3_at_origin"] = base_df[TARGET_COLUMN]

# Retardos recientes respecto al origen de predicción.
# Por ejemplo, o3_lag_1h representa el valor de O3 una hora antes del origen t.
for lag in ORIGIN_LAG_HOURS:
    base_df[f"o3_lag_{lag}h"] = base_df[TARGET_COLUMN].shift(lag)

# Ventanas móviles calculadas hasta el origen.
# min_periods=1 permite resumir la información disponible aunque existan huecos.
for window in ROLLING_WINDOWS:
    rolling = base_df[TARGET_COLUMN].rolling(window=window, min_periods=1)
    base_df[f"o3_roll_mean_{window}h"] = rolling.mean()

    # La desviación estándar necesita al menos dos observaciones válidas.
    base_df[f"o3_roll_std_{window}h"] = (
        base_df[TARGET_COLUMN]
        .rolling(window=window, min_periods=2)
        .std()
    )

# Máximo móvil de 24 horas.
# Se conserva como indicador sencillo de episodios recientes elevados.
base_df["o3_roll_max_24h"] = (
    base_df[TARGET_COLUMN]
    .rolling(window=24, min_periods=1)
    .max()
)

# Proporción de ausencias en las últimas 24 horas.
rolling_valid_count_24h = (
    base_df[TARGET_COLUMN]
    .rolling(window=24, min_periods=1)
    .count()
)
base_df["o3_roll_missing_ratio_24h"] = 1 - (rolling_valid_count_24h / 24)

# Una vez generadas las variables derivadas, se elimina la columna original `o3`
# de este dataframe auxiliar para evitar ambigüedad con el objetivo `y`.
# El valor observado en el origen queda disponible como `o3_at_origin`.
base_df = base_df.drop(columns=[TARGET_COLUMN])

display(base_df.head(10))

,timestamp,o3_at_origin,o3_lag_1h,o3_lag_2h,o3_lag_3h,o3_lag_6h,o3_lag_12h,o3_roll_mean_6h,o3_roll_std_6h,o3_roll_mean_24h,o3_roll_std_24h,o3_roll_max_24h,o3_roll_missing_ratio_24h
0,2020-01-01 00:00:00,22.0,NaN,NaN,NaN,NaN,NaN,22.000000,NaN,22.000000,NaN,22.0,0.958333
1,2020-01-01 01:00:00,19.0,22.0,NaN,NaN,NaN,NaN,20.500000,2.121320,20.500000,2.121320,22.0,0.916667
2,2020-01-01 02:00:00,19.0,19.0,22.0,NaN,NaN,NaN,20.000000,1.732051,20.000000,1.732051,22.0,0.875000
3,2020-01-01 03:00:00,6.0,19.0,19.0,22.0,NaN,NaN,16.500000,7.141428,16.500000,7.141428,22.0,0.833333
4,2020-01-01 04:00:00,8.0,6.0,19.0,19.0,NaN,NaN,14.800000,7.259477,14.800000,7.259477,22.0,0.791667
5,2020-01-01 05:00:00,5.0,8.0,6.0,19.0,NaN,NaN,13.166667,7.626707,13.166667,7.626707,22.0,0.750000
6,2020-01-01 06:00:00,4.0,5.0,8.0,6.0,22.0,NaN,10.166667,6.968979,11.857143,7.776644,22.0,0.708333
7,2020-01-01 07:00:00,2.0,4.0,5.0,8.0,19.0,NaN,7.333333,6.055301,10.625000,7.998884,22.0,0.666667
8,2020-01-01 08:00:00,2.0,2.0,4.0,5.0,19.0,NaN,4.500000,2.345208,9.666667,8.015610,22.0,0.625000
9,2020-01-01 09:00:00,8.0,2.0,2.0,4.0,6.0,NaN,4.833333,2.714160,9.500000,7.575545,22.0,0.583333


## 4. Construcción de los datasets de modelado por horizonte

Para cada horizonte `h`, se genera un dataset con:

- `origin_timestamp`: el instante desde el que se predice
- `target_timestamp`: el instante que se quiere predecir
- `y`: el valor observado de O3 en el instante objetivo
- Las variables de calendario del instante objetivo
- Las variables de historia de O3 disponibles en el origen
- `o3_prev_day_same_hour`, que coincide con la predicción puntual del modelo base ingenuo estacional
- `o3_target_lag_48h`, que incorpora la misma franja horaria de dos días antes

La columna `o3_prev_day_same_hour` es especialmente importante, ya que, para cada objetivo, contiene el valor de O3 registrado en la misma hora del día anterior. Por tanto, más adelante se podrá usar directamente como predicción del modelo base.

In [8]:

# Serie auxiliar para recuperar valores de O3 en función del timestamp.
o3_by_timestamp = (
    hourly_df
    .set_index("timestamp")[TARGET_COLUMN]
    .sort_index()
)

def build_modeling_dataset_for_horizon(
    base_features_df: pd.DataFrame,
    horizon: int,
    o3_series: pd.Series,
) -> pd.DataFrame:
    """Construye el dataset de modelado para un horizonte concreto.

    La fila asociada al origen t se utiliza para predecir y_{t+h}. Las variables
    basadas en O3 solo pueden proceder de t o de instantes anteriores.
    """
    df = base_features_df.copy()

    df = df.rename(columns={"timestamp": "origin_timestamp"})
    df["horizon"] = horizon
    df["target_timestamp"] = df["origin_timestamp"] + pd.to_timedelta(horizon, unit="h")

    # Objetivo: concentración de O3 en el instante a predecir.
    df["y"] = o3_series.reindex(df["target_timestamp"]).to_numpy()

    # Valor de la misma hora del día anterior al objetivo.
    # Esta columna coincide con la regla de persistencia diaria del modelo base.
    prev_day_timestamp = df["target_timestamp"] - pd.Timedelta(hours=24)
    df["o3_prev_day_same_hour"] = o3_series.reindex(prev_day_timestamp).to_numpy()

    # Retardo diario adicional respecto al objetivo: dos días antes.
    # Para los horizontes 1, 4, 12 y 24, este valor siempre queda antes o en el origen.
    for lag in TARGET_LAG_HOURS:
        source_timestamp = df["target_timestamp"] - pd.to_timedelta(lag, unit="h")
        column_name = f"o3_target_lag_{lag}h"
        df[column_name] = o3_series.reindex(source_timestamp).to_numpy()

    # Variables de calendario del instante objetivo.
    # Son válidas porque el calendario futuro se conoce antes de predecir.
    df["target_hour"] = df["target_timestamp"].dt.hour
    df["target_dayofweek"] = df["target_timestamp"].dt.dayofweek
    df["target_month"] = df["target_timestamp"].dt.month
    df["target_is_weekend"] = df["target_dayofweek"].isin([5, 6]).astype(int)

    df = add_cyclical_encoding(df, "target_hour", 24, "target_hour")
    df = add_cyclical_encoding(df, "target_dayofweek", 7, "target_dayofweek")

    # Para el mes se resta 1 antes de codificar, de modo que enero empiece en 0.
    df["target_month_zero_based"] = df["target_month"] - 1
    df = add_cyclical_encoding(df, "target_month_zero_based", 12, "target_month")

    # Variable auxiliar para documentar si el objetivo está ausente.
    df["target_is_missing"] = df["y"].isna().astype(int)

    # Partición cronológica según el instante objetivo.
    df["split"] = df["target_timestamp"].apply(assign_chronological_split)

    # Se descartan filas cuyo objetivo queda fuera del periodo 2020-2025.
    df = df[df["split"].notna()].copy()

    # Orden de columnas para facilitar la inspección.
    metadata_columns = [
        "horizon",
        "origin_timestamp",
        "target_timestamp",
        "split",
        "y",
        "target_is_missing",
    ]

    remaining_columns = [column for column in df.columns if column not in metadata_columns]
    df = df[metadata_columns + remaining_columns]

    return df

In [9]:

raw_datasets_by_horizon: dict[int, pd.DataFrame] = {}
modeling_datasets_by_horizon: dict[int, pd.DataFrame] = {}

summary_tables = []
missing_feature_tables = []

for horizon in HORIZONS:
    raw_horizon_df = build_modeling_dataset_for_horizon(
        base_features_df=base_df,
        horizon=horizon,
        o3_series=o3_by_timestamp,
    )

    check_no_temporal_leakage(raw_horizon_df, horizon=horizon)

    # Resumen antes de eliminar objetivos ausentes.
    summary_tables.append(summarize_by_split(raw_horizon_df, horizon=horizon))

    # Para entrenar y evaluar modelos no se pueden usar filas sin objetivo observado.
    # Sin embargo, los valores ausentes en las variables predictoras se conservan.
    modeling_horizon_df = raw_horizon_df.dropna(subset=["y"]).copy()

    raw_datasets_by_horizon[horizon] = raw_horizon_df
    modeling_datasets_by_horizon[horizon] = modeling_horizon_df

summary_df = pd.concat(summary_tables, ignore_index=True)

display(summary_df)

,horizon,split,n_rows,origin_min,origin_max,target_min,target_max,missing_target,missing_baseline_daily
0,1,train,26303,2020-01-01 00:00:00,2022-12-31 22:00:00,2020-01-01 01:00:00,2022-12-31 23:00:00,787,810
1,1,calibration,8760,2022-12-31 23:00:00,2023-12-31 22:00:00,2023-01-01 00:00:00,2023-12-31 23:00:00,153,153
2,1,validation,8784,2023-12-31 23:00:00,2024-12-31 22:00:00,2024-01-01 00:00:00,2024-12-31 23:00:00,347,347
3,1,test,8760,2024-12-31 23:00:00,2025-12-31 22:00:00,2025-01-01 00:00:00,2025-12-31 23:00:00,379,379
4,4,train,26300,2020-01-01 00:00:00,2022-12-31 19:00:00,2020-01-01 04:00:00,2022-12-31 23:00:00,787,807
5,4,calibration,8760,2022-12-31 20:00:00,2023-12-31 19:00:00,2023-01-01 00:00:00,2023-12-31 23:00:00,153,153
6,4,validation,8784,2023-12-31 20:00:00,2024-12-31 19:00:00,2024-01-01 00:00:00,2024-12-31 23:00:00,347,347
7,4,test,8760,2024-12-31 20:00:00,2025-12-31 19:00:00,2025-01-01 00:00:00,2025-12-31 23:00:00,379,379
8,12,train,26292,2020-01-01 00:00:00,2022-12-31 11:00:00,2020-01-01 12:00:00,2022-12-31 23:00:00,787,799
9,12,calibration,8760,2022-12-31 12:00:00,2023-12-31 11:00:00,2023-01-01 00:00:00,2023-12-31 23:00:00,153,153


## 5. Selección compacta de columnas predictoras

Se define una lista común de **21 variables candidatas**.

Las columnas se agrupan en cinco familias:

1. **Calendario**: hora, día de la semana y mes codificados con seno-coseno.
2. **Retardos recientes**: valores pasados de O3 respecto al origen de predicción.
3. **Retardos diarios alineados con el objetivo**: misma hora del día anterior y de dos días antes.
4. **Ventanas móviles**: nivel y variabilidad reciente.
5. **Calidad de la ventana reciente**: proporción de ausencias en las últimas 24 horas.

In [10]:
# 1) Variables de calendario codificadas de forma cíclica.
calendar_feature_columns = [
    "target_hour_sin",
    "target_hour_cos",
    "target_dayofweek_sin",
    "target_dayofweek_cos",
    "target_month_sin",
    "target_month_cos",
    "target_is_weekend",
]

# 2) Valor de O3 en el origen de predicción.
# Es información disponible en el momento t y resulta especialmente relevante
# para horizontes cortos.
origin_value_feature_columns = ["o3_at_origin"]

# 3) Retardos recientes respecto al origen de predicción.
origin_lag_feature_columns = [f"o3_lag_{lag}h" for lag in ORIGIN_LAG_HOURS]

# 4) Retardos diarios alineados con el objetivo.
# o3_prev_day_same_hour equivale a target_timestamp - 24h y coincide con el modelo base.
target_lag_feature_columns = [
    "o3_prev_day_same_hour",
    "o3_target_lag_48h",
]

# 5) Ventanas móviles.
rolling_feature_columns = [
    "o3_roll_mean_6h",
    "o3_roll_mean_24h",
    "o3_roll_std_6h",
    "o3_roll_std_24h",
    "o3_roll_max_24h",
]

# 6) Calidad de la ventana reciente.
quality_feature_columns = [
    "o3_roll_missing_ratio_24h",
]

candidate_feature_columns = (
    calendar_feature_columns
    + origin_value_feature_columns
    + origin_lag_feature_columns
    + target_lag_feature_columns
    + rolling_feature_columns
    + quality_feature_columns
)

assert len(candidate_feature_columns) == 21, f"Se esperaban 21 variables candidatas y se han definido {len(candidate_feature_columns)}."

# Se comprueba que todas las variables existan en todos los datasets.
for horizon, df in modeling_datasets_by_horizon.items():
    missing_features = [column for column in candidate_feature_columns if column not in df.columns]
    if missing_features:
        raise ValueError(f"Faltan columnas predictoras en el horizonte {horizon}: {missing_features}")

# Eliminación de duplicados exactos por horizonte.
# Se mantiene el conjunto común, pero se evitan redundancias cuando
# dos columnas contienen exactamente la misma información en un horizonte concreto.
feature_columns_by_horizon: dict[str, list[str]] = {}
dropped_duplicate_features_by_horizon: dict[str, dict[str, str]] = {}

for horizon, df in modeling_datasets_by_horizon.items():
    selected_features, dropped_duplicates = remove_exact_duplicate_features(
        df=df,
        candidate_features=candidate_feature_columns,
    )
    feature_columns_by_horizon[str(horizon)] = selected_features
    dropped_duplicate_features_by_horizon[str(horizon)] = dropped_duplicates

    print(f"Horizonte {horizon:02d}h: {len(selected_features)} variables utilizadas.")
    if dropped_duplicates:
        print("  Duplicados eliminados:", dropped_duplicates)

# Resumen de valores ausentes en las variables predictoras seleccionadas por horizonte.
missing_feature_tables = []
for horizon, df in modeling_datasets_by_horizon.items():
    selected_features = feature_columns_by_horizon[str(horizon)]
    feature_missing = (
        df[selected_features]
        .isna()
        .sum()
        .rename("n_missing")
        .reset_index()
        .rename(columns={"index": "column"})
    )
    feature_missing["horizon"] = horizon
    feature_missing["missing_ratio"] = feature_missing["n_missing"] / len(df)
    missing_feature_tables.append(feature_missing)

missing_features_df = pd.concat(missing_feature_tables, ignore_index=True)

display(missing_features_df.sort_values(["horizon", "missing_ratio"], ascending=[True, False]).head(80))

Horizonte 01h: 21 variables utilizadas.
Horizonte 04h: 21 variables utilizadas.
Horizonte 12h: 20 variables utilizadas.
  Duplicados eliminados: {'o3_prev_day_same_hour': 'o3_lag_12h'}
Horizonte 24h: 20 variables utilizadas.
  Duplicados eliminados: {'o3_prev_day_same_hour': 'o3_at_origin'}


,column,n_missing,horizon,missing_ratio
14,o3_target_lag_48h,1379,1,0.027071
13,o3_prev_day_same_hour,1048,1,0.020573
12,o3_lag_12h,812,1,0.015940
11,o3_lag_6h,660,1,0.012956
10,o3_lag_3h,550,1,0.010797
9,o3_lag_2h,472,1,0.009266
8,o3_lag_1h,365,1,0.007165
7,o3_at_origin,218,1,0.004279
17,o3_roll_std_6h,96,1,0.001885
18,o3_roll_std_24h,46,1,0.000903


In [11]:
# Catálogo de variables para documentar el dataset de modelado.
feature_catalog_rows = []

for column in calendar_feature_columns:
    feature_catalog_rows.append(
        {
            "feature": column,
            "family": "calendario",
            "uses_o3_history": False,
            "description": "Variable de calendario del instante objetivo con codificación cíclica o indicador de fin de semana.",
        }
    )

for column in origin_value_feature_columns:
    feature_catalog_rows.append(
        {
            "feature": column,
            "family": "valor_origen",
            "uses_o3_history": True,
            "description": "Valor de O3 observado en el origen de predicción t. Es información disponible antes de predecir y_{t+h}.",
        }
    )

for column in origin_lag_feature_columns:
    feature_catalog_rows.append(
        {
            "feature": column,
            "family": "retardo_reciente",
            "uses_o3_history": True,
            "description": "Retardo de O3 calculado respecto al origen de predicción.",
        }
    )

for column in target_lag_feature_columns:
    feature_catalog_rows.append(
        {
            "feature": column,
            "family": "retardo_diario_objetivo",
            "uses_o3_history": True,
            "description": "Retardo diario de O3 alineado con el instante objetivo y construido sin usar información futura.",
        }
    )

for column in rolling_feature_columns:
    feature_catalog_rows.append(
        {
            "feature": column,
            "family": "ventana_movil",
            "uses_o3_history": True,
            "description": "Resumen móvil calculado con valores de O3 disponibles hasta el origen de predicción.",
        }
    )

for column in quality_feature_columns:
    feature_catalog_rows.append(
        {
            "feature": column,
            "family": "calidad_ventana",
            "uses_o3_history": True,
            "description": "Proporción de valores ausentes en la ventana reciente de 24 horas.",
        }
    )

feature_catalog_df = pd.DataFrame(feature_catalog_rows)
display(feature_catalog_df)

print(f"Variables candidatas compactas: {len(candidate_feature_columns)}")

,feature,family,uses_o3_history,description
0,target_hour_sin,calendario,False,Variable de calendario del instante objetivo c...
1,target_hour_cos,calendario,False,Variable de calendario del instante objetivo c...
2,target_dayofweek_sin,calendario,False,Variable de calendario del instante objetivo c...
3,target_dayofweek_cos,calendario,False,Variable de calendario del instante objetivo c...
4,target_month_sin,calendario,False,Variable de calendario del instante objetivo c...
5,target_month_cos,calendario,False,Variable de calendario del instante objetivo c...
6,target_is_weekend,calendario,False,Variable de calendario del instante objetivo c...
7,o3_at_origin,valor_origen,True,Valor de O3 observado en el origen de predicci...
8,o3_lag_1h,retardo_reciente,True,Retardo de O3 calculado respecto al origen de ...
9,o3_lag_2h,retardo_reciente,True,Retardo de O3 calculado respecto al origen de ...


Variables candidatas compactas: 21


## 6. Guardado de datasets y tablas de control

Se guardan:

- Un archivo por horizonte (`modeling_h01.parquet`, etc.)
- Un archivo combinado (`modeling_all_horizons.parquet`)
- La lista de variables predictoras (`feature_columns.json`)
- Una tabla resumen de particiones y objetivos ausentes
- Un catálogo de variables
- Una tabla de valores ausentes por variable predictora

Estos archivos serán la base común para los notebooks de entrenamiento del modelo base, la regresión lineal, LightGBM y CatBoost.

In [12]:
combined_modeling_df = pd.concat(
    modeling_datasets_by_horizon.values(),
    ignore_index=True,
)

# Se guardan los datasets por horizonte.
for horizon, df in modeling_datasets_by_horizon.items():
    output_path = MODELING_DATA_DIR / f"modeling_h{horizon:02d}.parquet"
    df.to_parquet(output_path, index=False)
    print(f"Guardado: {output_path} ({len(df):,} filas)")

# Se guarda el dataset combinado.
combined_output_path = MODELING_DATA_DIR / "modeling_all_horizons.parquet"
combined_modeling_df.to_parquet(combined_output_path, index=False)
print(f"Guardado: {combined_output_path} ({len(combined_modeling_df):,} filas)")

# Se guarda la lista de variables predictoras.
feature_columns_path = MODELING_DATA_DIR / "feature_columns.json"
feature_columns_payload = {
    "target_column": "y",
    "candidate_feature_columns": candidate_feature_columns,
    "calendar_feature_columns": calendar_feature_columns,
    "origin_value_feature_columns": origin_value_feature_columns,
    "origin_lag_feature_columns": origin_lag_feature_columns,
    "target_lag_feature_columns": target_lag_feature_columns,
    "rolling_feature_columns": rolling_feature_columns,
    "quality_feature_columns": quality_feature_columns,
    "feature_columns_by_horizon": feature_columns_by_horizon,
    "dropped_duplicate_features_by_horizon": dropped_duplicate_features_by_horizon,
    "n_candidate_features": len(candidate_feature_columns),
}
feature_columns_path.write_text(
    json.dumps(feature_columns_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Guardado: {feature_columns_path}")

# Se guardan las tablas de control.
summary_path = REPORTS_TABLES_DIR / "modeling_dataset_summary.csv"
feature_catalog_path = REPORTS_TABLES_DIR / "modeling_feature_catalog.csv"
missing_features_path = REPORTS_TABLES_DIR / "modeling_feature_missing_summary.csv"

summary_df.to_csv(summary_path, index=False)
feature_catalog_df.to_csv(feature_catalog_path, index=False)
missing_features_df.to_csv(missing_features_path, index=False)

print(f"Guardado: {summary_path}")
print(f"Guardado: {feature_catalog_path}")
print(f"Guardado: {missing_features_path}")

# Se genera una muestra pequeña para realizar una inspección rápida. No se utilizará para ningún entrenamiento.
sample_path = REPORTS_TABLES_DIR / "modeling_h01_sample.csv"
modeling_datasets_by_horizon[1].head(200).to_csv(sample_path, index=False)
print(f"Guardado: {sample_path}")

Guardado: C:\trabajo_github\data\modeling\modeling_h01.parquet (50,941 filas)
Guardado: C:\trabajo_github\data\modeling\modeling_h04.parquet (50,938 filas)
Guardado: C:\trabajo_github\data\modeling\modeling_h12.parquet (50,930 filas)
Guardado: C:\trabajo_github\data\modeling\modeling_h24.parquet (50,918 filas)
Guardado: C:\trabajo_github\data\modeling\modeling_all_horizons.parquet (203,727 filas)
Guardado: C:\trabajo_github\data\modeling\feature_columns.json
Guardado: C:\trabajo_github\reports\tables\modeling_dataset_summary.csv
Guardado: C:\trabajo_github\reports\tables\modeling_feature_catalog.csv
Guardado: C:\trabajo_github\reports\tables\modeling_feature_missing_summary.csv
Guardado: C:\trabajo_github\reports\tables\modeling_h01_sample.csv


## 7. Comprobaciones finales

Antes de pasar al entrenamiento, se revisa que:

- Existan los datasets esperados
- Cada horizonte contenga las cuatro particiones cronológicas
- El conjunto de prueba quede identificado, si bien no se utilizará todavía
- No se hayan eliminado filas por valores ausentes en las variables predictoras, solo por ausencia del objetivo
- El conjunto compacto mantenga 21 variables candidatas

In [13]:

expected_dataset_files = [
    MODELING_DATA_DIR / f"modeling_h{horizon:02d}.parquet"
    for horizon in HORIZONS
]
expected_dataset_files.append(MODELING_DATA_DIR / "modeling_all_horizons.parquet")
expected_dataset_files.append(MODELING_DATA_DIR / "feature_columns.json")

missing_files = [path for path in expected_dataset_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"No se han generado los siguientes archivos esperados: {missing_files}")

# Resumen final por horizonte y partición del dataset ya utilizable para modelado.
final_summary_rows = []
for horizon, df in modeling_datasets_by_horizon.items():
    selected_features = feature_columns_by_horizon[str(horizon)]

    for split_name, split_df in df.groupby("split", sort=False):
        final_summary_rows.append(
            {
                "horizon": horizon,
                "split": split_name,
                "n_rows_modeling": len(split_df),
                "target_min": split_df["target_timestamp"].min(),
                "target_max": split_df["target_timestamp"].max(),
                "n_features_used": len(selected_features),
                "n_missing_features_total": split_df[selected_features].isna().sum().sum(),
                "n_missing_baseline_daily": split_df["o3_prev_day_same_hour"].isna().sum(),
            }
        )

final_summary_df = pd.DataFrame(final_summary_rows)
display(final_summary_df)

required_splits = set(SPLIT_DATE_RANGES.keys())
for horizon, df in modeling_datasets_by_horizon.items():
    observed_splits = set(df["split"].unique())
    missing_splits = required_splits - observed_splits
    if missing_splits:
        raise AssertionError(f"El horizonte {horizon} no contiene las particiones: {missing_splits}")

if len(candidate_feature_columns) != 21:
    raise AssertionError("El conjunto no contiene las 21 variables esperadas.")

print("Las comprobaciones finales se han superado con éxito. Los datasets de modelado por horizonte están listos para el entrenamiento.")

,horizon,split,n_rows_modeling,target_min,target_max,n_features_used,n_missing_features_total,n_missing_baseline_daily
0,1,train,25516,2020-01-01 01:00:00,2022-12-31 23:00:00,21,2577,470
1,1,calibration,8607,2023-01-01 00:00:00,2023-12-31 23:00:00,21,797,133
2,1,validation,8437,2024-01-01 00:00:00,2024-12-31 23:00:00,21,916,172
3,1,test,8381,2025-01-01 00:00:00,2025-12-31 23:00:00,21,1436,273
4,4,train,25513,2020-01-01 04:00:00,2022-12-31 23:00:00,21,3088,467
5,4,calibration,8607,2023-01-01 00:00:00,2023-12-31 23:00:00,21,964,133
6,4,validation,8437,2024-01-01 00:00:00,2024-12-31 23:00:00,21,1139,172
7,4,test,8381,2025-01-01 00:00:00,2025-12-31 23:00:00,21,1835,273
8,12,train,25505,2020-01-01 12:00:00,2022-12-31 23:00:00,20,3486,459
9,12,calibration,8607,2023-01-01 00:00:00,2023-12-31 23:00:00,20,984,133


Las comprobaciones finales se han superado con éxito. Los datasets de modelado por horizonte están listos para el entrenamiento.
